# 容器与对象引用

学习目标：选择并操作常用容器，理解共享引用、可变性与可哈希性，区分浅拷贝和深拷贝。

前置知识：变量与类型、基本运算、字符串索引与切片、方法调用。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 常用容器总览

容器把多个对象组织在一起。先根据“按位置读取、按键查找，还是去重与集合运算”选择类型。

本章的 list、tuple、range、dict、set 和 frozenset 都是 Python 内置容器类型；类型总览用于认识它们，本章进一步学习操作方式与引用行为。

| 类型 | 中文名称／含义 | 组织方式 | 可变性 |
| --- | --- | --- | --- |
| list | 列表 | 按位置存放元素，允许重复 | 可变 |
| tuple | 元组 | 按位置组合元素，允许重复 | 不可变 |
| range | 整数序列 | 按起点、终点和步长表示整数 | 不可变 |
| dict | 字典 | 键与值对应，保留键的插入顺序 | 可变 |
| set | 集合 | 元素不重复，不记录位置或插入顺序 | 可变 |
| frozenset | 不可变集合 | 元素不重复，不记录位置或插入顺序 | 不可变 |

这里的“不可变”约束容器自身，不保证它引用的对象也不可变。len 返回容器中的元素个数；字典按键值对计数。

In [1]:
print(type([]), type(()), type(range(3)))  # 分别为 list、tuple、range。
print(type({}), type(set()), type(frozenset()))  # 分别为 dict、set、frozenset。
print(len(["Python", "Java"]), len({"课程": "Python"}))  # 预期：2 1
print(len({"Python", "Python", "Java"}))  # 预期：2；重复元素只保留一份。

<class 'list'> <class 'tuple'> <class 'range'>
<class 'dict'> <class 'set'> <class 'frozenset'>
2 1
2


## 2 列表、元组与 range

### 2.1 列表的读取与修改

list 用方括号创建，元素可以是不同类型。索引、负索引和切片的读取规则与字符串相同。

列表允许按索引赋值，也能用切片替换一段内容。普通切片赋值可以改变列表长度；使用不为 1 的步长时，新元素数量必须与被替换的位置数量一致。单个索引赋值不能越界追加元素。

In [2]:
scores = [60, 70, 80, 90]
print(scores[0], scores[-1], scores[1:3])  # 预期：60 90 [70, 80]
scores[1] = 75
scores[2:4] = [85]
print(scores)  # 预期：[60, 75, 85]；切片替换让长度减少了一项。

scores[::2] = [61, 86]  # 索引 0 和 2 共两个位置，需要提供两个值。
print(scores)  # 预期：[61, 75, 86]

60 90 [70, 80]
[60, 75, 85]
[61, 75, 86]


### 2.2 添加元素

| 方法 | 中文名称／含义 |
| --- | --- |
| list.append | 在末尾添加一个对象 |
| list.extend | 在末尾逐项加入另一个可迭代对象的元素 |
| list.insert | 在指定索引之前插入一个对象 |

可迭代对象是可以逐项取出内容的对象，例如字符串、列表和元组。append 把传入的列表作为一项加入，extend 则取出其中各项加入。

这些方法原地修改列表，返回 None。不要把它们的返回值当作修改后的列表。

In [3]:
tasks = ["阅读"]
returned = tasks.append(["练习", "复习"])
print(tasks)  # 预期：['阅读', ['练习', '复习']]；嵌套列表是一项。
print(returned)  # 预期：None

tasks = ["阅读"]
tasks.extend(["练习", "复习"])
tasks.insert(1, "运行示例")
print(tasks)  # 预期：['阅读', '运行示例', '练习', '复习']

['阅读', ['练习', '复习']]
None
['阅读', '运行示例', '练习', '复习']


### 2.3 查找与删除

| 方法或语句 | 中文名称／含义 |
| --- | --- |
| list.index | 查找第一个匹配值的索引 |
| list.count | 统计匹配值的个数 |
| list.remove | 删除第一个匹配值 |
| list.pop | 按索引删除并返回元素，默认处理末项 |
| del | 删除指定位置或切片，不返回被删元素 |
| list.clear | 清空列表 |

index 和 remove 找不到值时触发 ValueError；pop 的索引越界或列表为空时触发 IndexError。删除前区分“按值删除”和“按位置删除”。

In [4]:
items = ["A", "B", "A", "C"]
print(items.index("A"), items.count("A"))  # 预期：0 2
items.remove("A")
removed = items.pop()
print(removed, items)  # 预期：C ['B', 'A']
del items[:1]
print(items)  # 预期：['A']
items.clear()
print(items)  # 预期：[]

0 2
C ['B', 'A']
['A']
[]


### 2.4 元组与解包

tuple 用逗号组合元素。只有一个元素时仍需逗号，例如 ("Python",)；("Python") 只是带括号的字符串，() 才是空元组。

解包把各项分配给多个名称。没有星号时，名称数量必须与元素数量一致；同一层可用一个带 \* 的名称收集剩余元素，收集结果是列表。

元组不能替换、增加或删除自身的元素；内部可变对象的变化在第 5.2 节区分。

In [5]:
course = ("Python", 12)
name, lesson_count = course
print(name, lesson_count)  # 预期：Python 12
print(type(("Python",)), type(("Python")))  # 预期：tuple 与 str。

first, *middle, last = (10, 20, 30, 40)
print(first, middle, last)  # 预期：10 [20, 30] 40
left, right = 1, 2
left, right = right, left
print(left, right)  # 预期：2 1；先取得右侧的值，再完成赋值。

Python 12
<class 'tuple'> <class 'str'>
10 [20, 30] 40
2 1


### 2.5 range 表示整数序列

range(start, stop, step) 中，start 是起点，stop 是不包含在结果中的终点，step 是步长。只写一个参数时表示 stop；start 默认 0，step 默认 1，步长不能为 0。

range 保存范围信息，按需要计算元素，不预先创建包含全部整数的列表。它支持索引、切片与长度查询；下例用 list 转换小范围，方便观察。

In [6]:
print(list(range(4)))  # 预期：[0, 1, 2, 3]
print(list(range(2, 9, 2)))  # 预期：[2, 4, 6, 8]
print(list(range(5, 0, -2)))  # 预期：[5, 3, 1]
print(list(range(5, 0)))  # 预期：[]；正步长无法从 5 走向更小的终点。

numbers = range(10, 20, 2)
print(numbers[1], numbers[-1], len(numbers))  # 预期：12 18 5
print(numbers[1:3])  # 预期：range(12, 16, 2)；切片结果仍是 range。

[0, 1, 2, 3]
[2, 4, 6, 8]
[5, 3, 1]
[]
12 18 5
range(12, 16, 2)


### 2.6 排序与反转

| 方法或函数 | 中文名称／含义 |
| --- | --- |
| list.sort | 原地排序列表，返回 None |
| sorted | 返回新的排序列表，保留原容器 |
| list.reverse | 原地反转现有顺序，返回 None |

reverse=True 选择降序；key 指定取得排序依据的函数，例如 key=len 按元素长度排序。这里传入函数本身，不写 len()；函数对象在第 7 章展开。

sort 和 sorted 都是稳定排序：排序依据相等的元素保持原相对顺序。元素或 key 的结果必须能够比较大小，不能随意混排整数、字符串与 None。

In [7]:
values = [3, 1, 2]
ordered = sorted(values)
print(values, ordered)  # 预期：[3, 1, 2] [1, 2, 3]

returned = values.sort(reverse=True)
print(values, returned)  # 预期：[3, 2, 1] None
values.reverse()
print(values)  # 预期：[1, 2, 3]；反转当前顺序。

words = ["bb", "a", "cc"]
print(sorted(words, key=len))  # 预期：['a', 'bb', 'cc']；等长项顺序不变。

[3, 1, 2] [1, 2, 3]
[3, 2, 1] None
[1, 2, 3]
['a', 'bb', 'cc']


## 3 字典

### 3.1 按键读取与写入

dict 保存键值对，写作 {键: 值}。键必须可哈希，第 4.1 节解释这一条件；值可以是任意对象。重复写入同一个键会替换原值。

字典按键的插入顺序组织；更新已有键不会改变位置，删除后重新插入则放到末尾。in 检查键是否存在，不直接检查值。

用方括号读取不存在的键会触发 KeyError；get 可以提供默认值，且不会把缺失的键插入字典。

In [8]:
profile = {"name": "小林", "level": 1}
profile["level"] = 2
profile["course"] = "Python"
print(profile)  # 预期键顺序：name、level、course。
print(profile["name"], profile.get("city", "未填写"))  # 预期：小林 未填写
print("name" in profile, "小林" in profile)  # 预期：True False
print("city" in profile)  # 预期：False；get 没有插入键。

{'name': '小林', 'level': 2, 'course': 'Python'}
小林 未填写
True False
False


### 3.2 更新、合并与删除

| 方法或操作 | 中文名称／含义 |
| --- | --- |
| dict.update | 原地更新键值对，同键由新值覆盖，返回 None |
| dict.setdefault | 键存在时返回原值，否则插入并返回默认值 |
| dict.pop | 删除指定键并返回值，可提供缺失时的默认值 |
| del | 删除指定键值对，缺失时触发 KeyError |
| dict.clear | 清空字典 |
| 字典 \| 字典 | 合并为新字典，同键采用右侧值 |

pop 未提供默认值且键不存在时，也会触发 KeyError。get 只读取，setdefault 可能写入，使用时要区分。

In [9]:
settings = {"theme": "light", "size": 12}
settings.update({"size": 14})
print(settings.setdefault("theme", "dark"))  # 预期：light；已有值不会被覆盖。
settings.setdefault("language", "zh")
merged = settings | {"theme": "dark"}
print(settings["theme"], merged["theme"])  # 预期：light dark
print(settings.pop("size"), settings.pop("missing", None))  # 预期：14 None
del settings["language"]
print(settings)  # 预期：{'theme': 'light'}
settings.clear()
print(settings)  # 预期：{}。

light
light dark
14 None
{'theme': 'light'}
{}


### 3.3 字典视图与快照

| 方法 | 中文名称／含义 |
| --- | --- |
| dict.keys | 键视图 |
| dict.values | 值视图 |
| dict.items | 键值对视图，每项是一个二元组 |

视图随字典变化而更新，不是创建时的内容快照。用 list 可以收集当时的各项，但不会自动深拷贝其中的对象。

遍历字典或视图时增删键，可能触发 RuntimeError 或漏掉条目；第 5 章用新容器或快照避免这种修改方式。

In [10]:
scores = {"Ada": 80}
keys_view = scores.keys()
items_view = scores.items()
key_snapshot = list(keys_view)
scores["Bo"] = 90
scores["Ada"] = 85

print(list(keys_view))  # 预期：['Ada', 'Bo']
print(list(scores.values()))  # 预期：[85, 90]
print(list(items_view))  # 预期：[('Ada', 85), ('Bo', 90)]
print(key_snapshot)  # 预期：['Ada']；之前收集的键列表不随视图增加。

['Ada', 'Bo']
[85, 90]
[('Ada', 85), ('Bo', 90)]
['Ada']


## 4 可哈希性与集合

### 4.1 什么对象能作键或集合元素

可哈希（hashable）对象的哈希值在其生命周期内保持不变；比较相等的可哈希对象必须具有相同哈希值，反过来不一定成立。hash 可以取得哈希值，字典和集合用它辅助查找。

常见数值和字符串可哈希；list、dict、set 不可哈希。tuple 只有在每个元素都可哈希时才可哈希，所以“元组不可变”不等于“任何元组都能作键”。frozenset 的元素也必须可哈希。

可变性与可哈希性不是同一个概念，自定义对象的规则在对象模型章节展开。

In [11]:
locations = {(2, 3): "起点"}
print(locations[(2, 3)])  # 预期：起点；由整数组成的元组可作键。
print(hash((2, 3)) == hash((2, 3)))  # 预期：True；不依赖具体哈希数值。

numbers = {1: "整数"}
numbers[True] = "布尔值"
print(len(numbers), numbers[1])  # 预期：1 布尔值；True 与 1 相等，是同一个键。
# ([2, 3],) 含不可哈希的列表，不能作为普通 dict 的键。

起点
True
1 布尔值


### 4.2 set 与 frozenset

集合用于去重和成员检测，不支持按索引读取，也不保留插入顺序。{} 是空字典，空集合要写 set()；frozenset 创建后不能增删元素。

| set 方法 | 中文名称／含义 |
| --- | --- |
| set.add | 添加一个元素 |
| set.update | 加入可迭代对象中的各元素 |
| set.remove | 删除元素，不存在时触发 KeyError |
| set.discard | 删除元素，不存在时不报错 |
| set.pop | 删除并返回任意一个元素，空集合触发 KeyError |
| set.clear | 清空集合 |

下面把集合转换成排序列表，只是为了稳定地观察输出，不表示集合内部有排序。

In [12]:
tags = set(["Python", "SQL", "Python"])
tags.add("Git")
tags.update(["SQL", "Linux"])
tags.remove("Git")
tags.discard("不存在")
print(sorted(tags))  # 预期：['Linux', 'Python', 'SQL']

fixed = frozenset(tags)
print(type(fixed), len(fixed))  # 预期：<class 'frozenset'> 3
tags.clear()
print(len(tags), len(fixed))  # 预期：0 3；清空 tags 不影响已有的 fixed。

['Linux', 'Python', 'SQL']
<class 'frozenset'> 3
0 3


### 4.3 集合运算

下表中的 a、b 都表示集合；这些运算不会修改它们。

| 写法 | 中文名称／含义 |
| --- | --- |
| a \| b | 并集：任一集合中出现的元素 |
| a & b | 交集：两边共有的元素 |
| a - b | 差集：在 a 中而不在 b 中的元素 |
| a ^ b | 对称差集：只在其中一边的元素 |
| a <= b | a 是否为 b 的子集，允许相等 |
| a < b | a 是否为 b 的真子集，必须不相等 |
| a.isdisjoint(b) | 两个集合是否没有共同元素 |

\>=、\> 对应超集与真超集。集合的大小比较表示包含关系，不是按元素个数排序；运算符形式要求两侧都是集合对象。

In [13]:
a = {1, 2, 3}
b = {3, 4}
print(sorted(a | b))  # 预期：[1, 2, 3, 4]
print(sorted(a & b))  # 预期：[3]
print(sorted(a - b), sorted(a ^ b))  # 预期：[1, 2] [1, 2, 4]
print({1} <= a, a < a, a >= {1})  # 预期：True False True
print(a.isdisjoint({8, 9}))  # 预期：True

[1, 2, 3, 4]
[3]
[1, 2] [1, 2, 4]
True False True
True


## 5 对象引用与拷贝

### 5.1 名称绑定、身份与相等

赋值建立名称与对象的绑定，不复制对象。两个名称指向同一可变对象时，通过任一名称修改它，另一名称也会看到变化。

== 比较值是否相等，is 比较是否为同一对象；id 返回对象生命周期内的标识值。比较普通数值或字符串内容时使用 ==，不要依赖解释器是否复用对象。

del 名称只删除这个绑定，其他名称仍可引用该对象；这与 del 列表[索引] 修改容器不同。

In [14]:
original = [1, 2]
alias = original
equal_values = [1, 2]
print(original == equal_values, original is equal_values)  # 预期：True False
print(original is alias, id(original) == id(alias))  # 预期：True True

alias.append(3)
print(original)  # 预期：[1, 2, 3]
del original
print(alias)  # 预期：[1, 2, 3]；删除一个名称没有清空列表。

True False
True True
[1, 2, 3]
[1, 2, 3]


### 5.2 容器内的共享引用

容器保存对元素的引用。元组不能替换其中某个位置，但它引用的列表仍能修改。

列表重复和 dict.fromkeys 也可能让多个位置引用同一个可变对象。fromkeys 按给定的键创建字典，所有键使用同一个默认值对象；需要独立内容时分别创建，推导式写法在第 5 章介绍。

In [15]:
record = ("Python", ["阅读"])
record[1].append("练习")
print(record)  # 预期：('Python', ['阅读', '练习'])；元组仍引用原列表。

rows = [[]] * 2
rows[0].append("A")
print(rows, rows[0] is rows[1])  # 预期：[['A'], ['A']] True

groups = dict.fromkeys(["morning", "evening"], [])
groups["morning"].append("Ada")
print(groups)  # 预期：两个键对应的列表都包含 Ada。

('Python', ['阅读', '练习'])
[['A'], ['A']] True
{'morning': ['Ada'], 'evening': ['Ada']}


### 5.3 浅拷贝与深拷贝

浅拷贝创建新的外层容器，但内部元素仍可共享引用。列表的 copy、完整切片，以及字典的 copy 都属于浅拷贝。判断修改是否互相影响，要沿引用找到真正被修改的对象。

![浅拷贝：外层不同，内层仍可共享](image/illustration/04-01-shallow-deep-copy.svg)

图示：依据 Python copy 文档自行绘制的浅拷贝与深拷贝对照。箭头是引用，框的数量区分对象身份；这不是所有对象都可深拷贝的承诺。

标准库 copy.copy 提供通用浅拷贝，copy.deepcopy 递归处理内部对象。深拷贝会处理重复引用，自定义对象也能控制复制行为，并非所有资源都能复制，因此不能概括为“所有对象都完全独立”。

下面用 import copy 引入工具。先沿 scores 的箭头预测 append(95) 会影响哪些字典，再用输出与 is 核对；新增 note 则只改变 shallow 的外层字典。

In [16]:
import copy

source = {"scores": [80, 90]}
shallow = source.copy()
deep = copy.deepcopy(source)
shallow["scores"].append(95)
shallow["note"] = "副本新增"

print(source)  # 预期：{'scores': [80, 90, 95]}；内层列表被共享。
print(deep)  # 预期：{'scores': [80, 90]}；本例的内层列表已复制。
print(shallow is source)  # 预期：False；浅拷贝也创建了外层字典。
print(shallow["scores"] is source["scores"])  # 预期：True
print(copy.copy([1, 2]))  # 预期：[1, 2]；通用浅拷贝接口。

{'scores': [80, 90, 95]}


{'scores': [80, 90]}
False
True
[1, 2]


### 5.4 列表的 += 与普通加法

对于列表，+= 原地扩展已有对象，其他引用会看到新增元素；列表 + 列表构造新的列表，重新赋值后原来的其他绑定仍保留旧列表。

因此，不能把所有增强赋值都机械替换成普通运算再赋值。第 2 章中整数的 += 是重新绑定，具体行为由对象类型决定。

In [17]:
items = [1]
alias = items
items += [2]
print(items, alias, items is alias)  # 预期：[1, 2] [1, 2] True

items = items + [3]
print(items, alias, items is alias)  # 预期：[1, 2, 3] [1, 2] False

[1, 2] [1, 2] True
[1, 2, 3] [1, 2] False


## 6 综合应用：整理报名名单

合并两批报名者，去重后按姓名排序；将结果放入字典，并保留后续修改前的独立副本。

这里再次导入 copy，使示例所用工具在本单元内可见。

In [18]:
import copy

# 1. 用集合去重，再转换为排序列表。
first_batch = ["Bo", "Ada", "Bo"]
second_batch = ["Chen", "Ada"]
members = sorted(set(first_batch) | set(second_batch))

# 2. 保留独立记录后，再修改当前名单。
registration = {"course": "Python", "members": members}
saved_registration = copy.deepcopy(registration)
registration["members"].append("Dina")
print(saved_registration["members"])  # 预期：['Ada', 'Bo', 'Chen']
print(registration["members"])  # 预期：['Ada', 'Bo', 'Chen', 'Dina']

['Ada', 'Bo', 'Chen']
['Ada', 'Bo', 'Chen', 'Dina']


## 本章小结

（1）列表和元组按位置组织，字典按键组织，集合用于去重与集合运算；range 表示整数范围。

（2）区分原地修改与返回新对象，尤其注意 append、sort、update 等方法的返回值。

（3）可哈希性决定普通字典键和集合元素的资格；字典视图动态反映原字典。

（4）赋值不复制对象；浅拷贝复制外层，深拷贝按规则递归处理。修改前先判断哪些引用被共享。

## 练习

（1）将下列两批编号合并、去重并升序排列，结果保存为列表。保持两份输入列表的内容不变。

In [19]:
batch_a = [3, 1, 3]
batch_b = [2, 1, 4]

# 在此完成合并、去重和排序。
# 检查：结果为 [1, 2, 3, 4]，两份输入仍保留各自的原始内容。

（2）先预测输出，再运行。说明键视图、键列表和浅拷贝中的内层列表为什么表现不同。

In [20]:
config = {"tags": ["基础"]}
view = config.keys()
keys_before = list(view)
copied = config.copy()
config["enabled"] = True
copied["tags"].append("练习")

# 先记录预测，再检查外层结构与共享的内层对象。
print(list(view), keys_before)
print(config["tags"], copied["tags"])

['tags', 'enabled'] ['tags']
['基础', '练习'] ['基础', '练习']


（3）为两位学习者创建各自的配置副本，只给第一位添加“练习”标签；默认配置和第二位的标签都应保持不变。

In [21]:
defaults = {"tags": ["基础"], "limit": 10}

# 在此使用适当的复制方式，得到 learner_a 和 learner_b。
# 检查：第一位的标签为 ['基础', '练习']，另外两份仍为 ['基础']。
# 再修改 learner_a 的 limit，检查另外两份配置的 limit 仍为 10。

### 提示

第一题先取集合并集，再排序生成新列表。第三题从是否共享内层列表判断复制深度，不只比较外层字典是否相同。

### 参考解析

第一题用 sorted(set(batch_a) | set(batch_b)) 得到 [1, 2, 3, 4]，集合转换和 sorted 都产生新对象，不修改两个原列表。

第二题的动态视图转成列表后是 ['tags', 'enabled']，先前保存的键列表仍为 ['tags']；两个字典的 tags 都是 ['基础', '练习']。增加键改变原字典的视图，浅拷贝则只复制外层字典，内层列表仍共享。

第三题分别对 defaults 调用 copy.deepcopy，得到两个配置，再修改配置 A 的 tags。A 包含“基础”“练习”，B 与 defaults 仍只有“基础”。继续改变 A 的 limit 时，B 与 defaults 仍为 10；但这一项不能单独证明使用了深拷贝，因为浅拷贝得到的外层字典也能独立重新绑定 limit。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [教程 §5.1–5.5 数据结构](https://docs.python.org/zh-cn/3.12/tutorial/datastructures.html)：列表、del、元组、解包、集合与字典；[可变序列](https://docs.python.org/zh-cn/3.12/library/stdtypes.html#typesseq-mutable)、[range](https://docs.python.org/zh-cn/3.12/library/stdtypes.html#ranges)、[字典](https://docs.python.org/zh-cn/3.12/library/stdtypes.html#typesmapping)、[字典视图](https://docs.python.org/zh-cn/3.12/library/stdtypes.html#dictionary-view-objects)、[集合类型](https://docs.python.org/zh-cn/3.12/library/stdtypes.html#set-types-set-frozenset)：容器行为与边界；[list.sort](https://docs.python.org/zh-cn/3.12/library/stdtypes.html#list.sort)及 [sorted](https://docs.python.org/zh-cn/3.12/library/functions.html#sorted)：排序依据与稳定性；[可哈希术语](https://docs.python.org/zh-cn/3.12/glossary.html#term-hashable)及 [hash](https://docs.python.org/zh-cn/3.12/library/functions.html#hash)：可哈希性与相等约束；[数据模型 §3.1](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#objects-values-and-types)及 [id](https://docs.python.org/zh-cn/3.12/library/functions.html#id)：身份、可变性和引用；[赋值与增强赋值 §7.2](https://docs.python.org/zh-cn/3.12/reference/simple_stmts.html#assignment-statements)：名称绑定、星号解包和原地操作；[copy](https://docs.python.org/zh-cn/3.12/library/copy.html)：浅拷贝、深拷贝及限制。 |